In [ ]:
from metadata import *
from preprocessing import *
from microfilm.microplot import microshow
import matplotlib.pyplot as plt
from calc_synaptic_coloc import *
from calc_synaptic_metrics import *

In [ ]:
path = "/Volumes/KINGSTON/code/phd/image-analysis/synapse-counting/test-images-VLGUT1-PSD95-A/OE_Exp1_IHC_Exp1_HA-GPR37L1_555-VGLUT1_647-PSD95_63X_airyscan_1.8zoom_CA1_SO.czi"
r, p, s = extract_metadata(path)
print(r, p, s)
image_filename(path, [0, 1, 2, 3, 7, 10, 11])

In [ ]:
pre, post = extract_and_split(path)

fig, axs = plt.subplots(1, 2, figsize = (30, 30))
microshow(pre, ax=axs[0], label_text = 'pre')
microshow(post, ax=axs[1], label_text = 'post')

In [ ]:
mfi_synapse(pre, post)

In [ ]:
puncta_results = puncta_metrics(pre, post, image_size_um = s, pixel_size_um = r)
puncta_results["pre_puncta_nr"]

In [ ]:
p = ImagePreprocessing(include_clahe=True, include_rolling_ball=True, include_tophat=True, include_blur=False,
                       clip_limit = 0.006, kernel_size = 200, nbins = 265)
pre_1, post_1 = p.preprocess(pre, post)
if np.array_equal(pre_1, pre):
    print("Yes, the arrays are the same.")
else:
    print("No, the arrays are not the same.")

fig, axs = plt.subplots(1, 2, figsize = (30, 30))
microshow(pre_1, ax=axs[0], label_text = 'pre')
microshow(post_1, ax=axs[1], label_text = 'post')


In [ ]:
p = ImagePreprocessing(include_clahe=False, include_rolling_ball=True, include_tophat=False, include_blur=True,
                       clip_limit = 0.006, kernel_size = 200, nbins = 265)
pre_2, post_2 = p.preprocess(pre, post)

mfi_synapse(pre_2,post_2)

In [ ]:
puncta_metrics(pre_1, post_1, image_size_um = s, pixel_size_um = r)

In [ ]:
pearsons_coloc(pre_1, post_1)
# manders_coloc(pre_1, post_1)
# overlap_um2_coloc(pre_1, post_1, pixel_size_in_um=r)


In [ ]:
import itertools
import matplotlib.pyplot as plt

# Assuming you have defined the ImagePreprocessing class and microshow function

# List all possible combinations of True and False for the parameters
combinations = list(itertools.product([True, False], repeat=4))

# Iterate through each combination
for idx, (include_clahe, include_rolling_ball, include_tophat, include_blur) in enumerate(combinations):
    # Create an instance of ImagePreprocessing with the current combination
    p = ImagePreprocessing(include_clahe=include_clahe, 
                           include_rolling_ball=include_rolling_ball,
                           include_tophat=include_tophat,
                           include_blur=include_blur)
    
    # Preprocess the images
    pre_processed, post_processed = p.preprocess(pre, post)
    
    # Show the pre-processed and post-processed images
    fig, axs = plt.subplots(1, 2, figsize=(30, 30))
    microshow(pre_processed, ax=axs[0], label_text='pre')
    microshow(post_processed, ax=axs[1], label_text='post')
    
    # Add a title to indicate the combination of preprocessing steps
    title = f"Combination {idx + 1}: CLAHE={include_clahe}, Rolling Ball={include_rolling_ball}, Top-hat={include_tophat}, Blur={include_blur}"
    plt.suptitle(title, fontsize=16)
    
    plt.show()



In [ ]:
import pandas as pd
import os

# the input folder
data_dir = "/Volumes/KINGSTON/code/phd/image-analysis/synapse-counting/output_data/"

# get list of files in the folder
all_files = []
for filename in os.listdir(data_dir):
  if filename.endswith(".csv"):
    all_files.append(os.path.join(data_dir, filename))

# read in the first .csv
base_df = pd.read_csv(all_files[0])

# setting the index for common column such that we can merge the following dataframes
base_df.set_index("img_filename", inplace=True)

# loop through remaining files and concatenate
for filename in all_files[1:]:
  df = pd.read_csv(filename, encoding = "utf-8")
  df.set_index("img_filename", inplace=True)
  base_df = pd.concat([base_df, df], axis=1)

# clean up merged dataframe
merged_df = base_df.drop(["Unnamed: 0"], axis=1)
merged_df = merged_df.reset_index()

# display merged dataframe
print(merged_df)


In [ ]:
# reading in the data
import pandas as pd
import glob

# the input folder
data_dir = "/Volumes/KINGSTON/code/phd/image-analysis/synapse-counting/intermediate_data/"

# all files in the folder
all_files = glob.glob(f"{data_dir}/*.csv")

# read in the first .csv
base_df = pd.read_csv(all_files[0])

# setting the index for common column such that we can merge the following dataframes
base_df.set_index("img_filename", inplace = True)
base_df

for filename in all_files[1:]:
  df = pd.read_csv(filename)
  df.set_index("img_filename", inplace = True)
  base_df = pd.concat([base_df, df], axis=1)
merged_df = base_df.drop(["Unnamed: 0"], axis = 1)
merged_df = merged_df.reset_index()

merged_df



In [ ]:
data = merged_df
data['gRNA'] = data['img_filename'].apply(lambda x: x.split('_')[-3:-2]).apply(lambda x: '_'.join(x))
data['hippocampal_layer'] = data['img_filename'].apply(lambda x: x.split('_')[-2:]).apply(lambda x: ' '.join(x))

data

In [ ]:
import seaborn as sns
p = sns.swarmplot(x="hippocampal_layer", y="overlap_um2", hue="gRNA",
                    data=data,
                    dodge=True,
                    marker="o",
                    alpha=0.5)
p

In [ ]:
def plot_data(df, x, y, extra_y_upper, title, hue = "gRNA", extra_y_lower=0, ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    p = sns.swarmplot(x=x, y=y, hue=hue,
                      data=df,
                      dodge=True,
                      marker="o",
                      alpha=0.5,
                      ax=ax)
    max_y = df[y].max()
    min_y = df[y].min()

    y_upper_limit = max_y + extra_y_upper

    if min_y < 0:
        y_lower_limit = min_y + extra_y_lower
    else:
        y_lower_limit = 0

    p.set_title(title)
    p.set_ylim(y_lower_limit, y_upper_limit)
    return p

# Create the figure object
fig, axes = plt.subplots(nrows=6, ncols=2, figsize=(15, 30))

# colocalization metrics
plot_overlap = plot_data(df=data, x="hippocampal_layer", y="overlap_um2", extra_y_upper=50, title='Overlap by hippocampal Layer and condition', ax=axes[0,0])
plot_pearson = plot_data(df=data, x="hippocampal_layer", y="pearson_cor", extra_y_upper=0.1, title='Pearsons correlation by hippocampal Layer and condition', extra_y_lower=-0.05, ax=axes[0,1])
plot_manders = plot_data(df=data, x="hippocampal_layer", y="overlap_coeff", extra_y_upper=0.1, title='Manders overlap coefficient by hippocampal Layer and condition', extra_y_lower=-0.05, ax=axes[1,0])

# single synapse marker metrics
plot_pre_mfi = plot_data(df=data, x="hippocampal_layer", y="presynapse_image_mfi", extra_y_upper=200, title='Presynaptic MFI by hippocampal Layer and condition', extra_y_lower=-0.05, ax=axes[2,0])
plot_pre_mfi = plot_data(df=data, x="hippocampal_layer", y="postsynapse_image_mfi", extra_y_upper=200, title='Postsynaptic MFI by hippocampal Layer and condition', extra_y_lower=-0.05, ax=axes[2,1])
plot_pre_density = plot_data(df=data, x="hippocampal_layer", y="pre_puncta_density_per_100_um2", extra_y_upper=100, title='Presynaptic puncta density MFI by hippocampal Layer and condition', extra_y_lower=-0.05, ax=axes[3,0])
plot_post_density = plot_data(df=data, x="hippocampal_layer", y="post_puncta_density_per_100_um2", extra_y_upper=100, title='Postsynaptic puncta density MFI by hippocampal Layer and condition', extra_y_lower=-0.05, ax=axes[3,1])
plot_pre_area = plot_data(df=data, x="hippocampal_layer", y="pre_staining_area_um2", extra_y_upper=50, title='Presynaptic staining area by hippocampal Layer and condition', extra_y_lower=-0.05, ax=axes[4,0])
plot_post_area = plot_data(df=data, x="hippocampal_layer", y="post_staining_area_um2", extra_y_upper=50, title='Postsynaptic staining area by hippocampal Layer and condition', extra_y_lower=-0.05, ax=axes[4,1])
plot_pre_puncta_size = plot_data(df=data, x="hippocampal_layer", y="pre_mean_puncta_size_um2", extra_y_upper=0.05, title='Presynaptic puncta area by hippocampal Layer and condition', extra_y_lower=-0.05, ax=axes[5,0])
plot_post_puncta_size = plot_data(df=data, x="hippocampal_layer", y="post_mean_puncta_size_um2", extra_y_upper=0.05, title='Postsynaptic puncta area by hippocampal Layer and condition', extra_y_lower=-0.05, ax=axes[5,1])



# Save the figure explicitly
combined_plots_figure = plt.gcf()  # Get the current figure object
plt.tight_layout()  # Adjust layout to prevent overlap
combined_plots_figure.savefig("combined_plots.png")  # Save the figure object
plt.show()

In [ ]:
# Columns you want to plot
columns_to_plot = ['overlap_um2', 'pearson_cor', 'overlap_coeff', "presynapse_image_mfi", "postsynapse_image_mfi", "pre_puncta_density_per_100_um2", "post_puncta_density_per_100_um2", "pre_staining_area_um2", "post_staining_area_um2", "pre_mean_puncta_size_um2", "post_mean_puncta_size_um2"]

# Loop through each column and create a swarmplot
for column in columns_to_plot:
    # Create the swarmplot
    p = sns.swarmplot(x="hippocampal_layer", y=column, hue="gRNA",
                      data=data,
                      dodge=True,
                      marker="o",
                      alpha=0.5)
    
    # Add title and labels
    plt.title(f'{column} by hippocampal layer')
    plt.xlabel('Hippocampal Layer')
    plt.ylabel(column)
    
    # Show the plot
    plt.show()

In [ ]:
df_melted_test = pd.melt(merged_df, id_vars=["img_filename"], value_vars=["overlap_um2", "overlap_um2_rot"], var_name = "condition", value_name="overlap in um2")
df_melted_test.head(10)

In [ ]:
def data_formatting(df, id_vars, value_vars, value_name, var_name = "condition"):
        """
        Wrapper function for formating the data for plotting into seaborn swarmplot.

        Args:
           df: the dataframe from which to plot
           id_vars: column with the image filename containing text for plotting on the x-axis (eg CA1 SR, CA1 SLM, ...)
           value_vars: columns to unpivot (see documentation pd.melt)
           value_name: the name on the y-axis
           var_name: the condition of rotated vs actual in synaptic marker colocalization

        Returns:
           df_melted: a dataframe pivoted and ready as input into seaborn swarmplot.
        """
        df_melted = pd.melt(df, id_vars = id_vars, value_vars = value_vars, var_name = var_name, value_name = value_name)
        df_melted["condition"] = df_melted[var_name].apply(lambda x: "rotated" if value_vars[1] in x else "actual")
        df_melted["hippocampal layer"] = df_melted[id_vars[0]].apply(lambda x: " ".join(x.split("_")[-2:]))
        return df_melted

In [ ]:
df_melted_overlap = data_formatting(df=merged_df, id_vars=["img_filename"], 
                                value_vars=["overlap_um2", "overlap_um2_rot"],
                                value_name="overlap in um2")
df_melted_overlap.head(10)

In [ ]:
df_melted_overlap = data_formatting(df=merged_df, id_vars=["img_filename"], 
                                value_vars=["overlap_um2", "overlap_um2_rot"],
                                value_name="overlap in um2")

In [ ]:
merged_df.columns

In [ ]:
df_melted_pearson = data_formatting(df=merged_df, id_vars=["img_filename"], 
                                value_vars=["pearson_cor", "pearson_cor_rot"],
                                value_name="pearsons correlation")
df_melted_pearson.head(10)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


def plot_data(df, x, y, extra_y_upper, title, extra_y_lower = 0, ax = None):
    if ax is None:
        fig, ax = plt.subplots()
    p = sns.swarmplot(x=x, y=y, hue = "condition",
                    data = df,
                    # jitter = False,
                    dodge = True,
                    marker = "o",
                    alpha = 0.5)
    max_y = df[y].max()
    min_y = df[y].min()

    y_upper_limit = max_y + extra_y_upper
    
    if min_y < 0:
        y_lower_limit = min_y + extra_y_lower
    else:
        y_lower_limit = 0
    
    p.set_title(title)
    p.set_ylim(y_lower_limit, y_upper_limit)
    return p

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_data(df, x, y, extra_y_upper, title, extra_y_lower=0, ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    p = sns.swarmplot(x=x, y=y, hue="condition",
                      data=df,
                      dodge=True,
                      marker="o",
                      alpha=0.5,
                      ax=ax)
    max_y = df[y].max()
    min_y = df[y].min()

    y_upper_limit = max_y + extra_y_upper

    if min_y < 0:
        y_lower_limit = min_y + extra_y_lower
    else:
        y_lower_limit = 0

    p.set_title(title)
    p.set_ylim(y_lower_limit, y_upper_limit)
    return p

# Create the figure object
fig, axes = plt.subplots(nrows=2, figsize=(8, 10))

plot_overlap = plot_data(df=df_melted_overlap, x="hippocampal layer", y="overlap in um2", extra_y_upper=100, title='Overlap by hippocampal Layer and condition', ax=axes[0])
plot_pearson = plot_data(df=df_melted_pearson, x="hippocampal layer", y="pearsons correlation", extra_y_upper=0.1, title='Pearsons correlation by hippocampal Layer and condition', extra_y_lower=-0.05, ax=axes[1])

# Save the figure explicitly
combined_plots_figure = plt.gcf()  # Get the current figure object
plt.tight_layout()  # Adjust layout to prevent overlap
combined_plots_figure.savefig("combined_plots.png")  # Save the figure object
plt.show()

